### Importing Libraries

In [10]:
import numpy as np 
import pandas as pd
import nltk
import string as s
import re
import matplotlib.pyplot as plt
import os
import string
#!pip install -U tensorflow
#!pip install -U tensorflow-text
import tensorflow as tf
import tensorflow_text as text
import tensorflow_hub as hub

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/priya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Read Data

In [11]:
data = pd.read_csv('Processed_data.csv',index_col=0)
X = data['headline']
y = data['clickbait']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size= 0.20)

### BERT

In [13]:
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4")

In [14]:
text_input = Input(shape=(), dtype=tf.string, name='text')
preprocessed_text = bert_preprocess(text_input)
outputs = bert_encoder(preprocessed_text)

KeyboardInterrupt: 

In [ ]:
l = Dropout(0.1, name="dropout")(outputs['pooled_output'])
l = Dense(1, activation='sigmoid', name="output")(l)
model = Model(inputs=[text_input], outputs = [l])
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 text (InputLayer)              [(None,)]            0           []                               
                                                                                                  
 keras_layer (KerasLayer)       {'input_mask': (Non  0           ['text[0][0]']                   
                                e, 128),                                                          
                                 'input_type_ids':                                                
                                (None, 128),                                                      
                                 'input_word_ids':                                                
                                (None, 128)}                                                  

In [ ]:
model.compile(optimizer='adam',
 loss='binary_crossentropy',
 metrics=['accuracy'])

In [ ]:
model.fit(X_train, y_train, batch_size=512, validation_data=(X_test, y_test), epochs=3)

 4/50 [=>............................] - ETA: 32:39 - loss: 0.7414 - accuracy: 0.4893

In [ ]:
y_predictions = model.predict(X_test)
y_predictions = y_predictions.flatten()
y_predictions = np.where(y_predictions > 0.5, 1, 0)

In [ ]:
def test_results(preds):
    return "Testing Accuracy:", accuracy_score(y_test,preds)," Testing Recall:", recall_score(y_test,preds), "F1 Score:",f1_score(y_test,preds) 

print(test_results(y_predictions))

In [ ]:
#confusion matrix on test set BERT C
sns.set()

cm_dc = confusion_matrix(y_test, rf_test_preds)
sns.heatmap(cm_dc.T, square=True, annot=True, fmt='d', cbar=False,cmap="inferno", xticklabels=['non-clickbait','clickbait'],yticklabels=['non-clickbait','clickbait']
            )
plt.xlabel('true label')
plt.ylabel('predicted label')